# Comparative gene regulatory network (GRN) analysis in AML/leukemic and immune cells

**Purpose.** Infer candidate transcription-factor (TF) → target-gene networks from a public, annotated single-cell RNA-seq dataset of primary acute myeloid leukemia (AML), then compare AML-like cells with immune-cell populations when cell annotations permit.

## Published data and benchmarking references

1. **Primary AML dataset:** Sandén et al., *Nature Cancer* (2025), “Aberrant expression of SLAMF6 constitutes a targetable immune escape mechanism in acute myeloid leukemia.” An annotated scRNA-seq count matrix (`.h5ad`) is made available via the SciLifeLab Data Repository (article DOI: [10.17044/scilifelab.28263911](https://doi.org/10.17044/scilifelab.28263911); [dataset page](https://figshare.scilifelab.se/articles/dataset/Single-cell_RNA_sequencing_data_on_primary_samples_from_Aberrant_expression_of_SLAMF6_constitutes_a_targetable_immune_escape_mechanism_in_acute_myeloid_leukemia/28263911)). The raw sequencing reads are access-controlled; use the publicly available annotated matrix.
2. **GRN benchmark methods:** Pratapa et al., *Nature Methods* (2020), “Benchmarking algorithms for gene regulatory network inference from single-cell transcriptomic data.” BEELINE provides benchmark expression data and reference networks at [Zenodo DOI 10.5281/zenodo.3378975](https://doi.org/10.5281/zenodo.3378975), and the framework/code at [GitHub](https://github.com/murali-group/BEELINE).
3. **Comparative network context:** Duren et al., *Nature Communications* (2021), [sc-compReg](https://doi.org/10.1038/s41467-021-25089-2), which illustrates disease-versus-control differential regulatory-network analysis in CLL and healthy B cells.

## What this notebook does

- Loads the annotated AML `.h5ad` matrix, inspects its cell annotations, and selects AML/leukemic and immune populations.
- Applies count normalization and log transformation when the expression matrix appears to contain raw counts.
- Selects shared variable genes and candidate human TFs, then infers group-specific TF→target scores with an **Extra-Trees / GENIE3-style feature-importance approach**.
- Exports edge lists, ranked regulators, differential-edge scores, and network plots.
- Optionally evaluates predictions against a user-supplied, compatible gold-standard TF-target edge list (for example, an appropriate BEELINE reference network).

> **Scientific caution:** expression-based network inference produces hypotheses, not proof of direct or causal regulation. A pooled “immune” network can be confounded by mixing cell lineages. Prefer one immune lineage at a time (for example, CD8 T cells or NK cells) and validate strong edges with independent evidence such as ChIP-seq, ATAC-seq/motif evidence, perturbation data, or published regulons. The optional BEELINE gold-standard evaluation is meaningful only when the benchmark expression data and ground truth correspond to the same dataset/organism and genes; a benchmark gold standard is not automatically an AML-specific validation set.

**Expected output directory:** `outputs/`

In [ ]:
# Install dependencies (run once per environment if required).
# In a managed environment you may instead install these in your conda/pip environment.
%pip -q install anndata numpy pandas scipy scikit-learn networkx matplotlib requests tqdm

In [ ]:
from pathlib import Path
import os, re, sys, json, zipfile, tarfile, warnings, shutil
import requests
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import networkx as nx
import anndata as ad
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import average_precision_score, roc_auc_score
from tqdm.auto import tqdm

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Public annotated AML scRNA-seq dataset (Nature Cancer, 2025).
FIGSHARE_ARTICLE_ID = 28263911
FIGSHARE_DOWNLOAD_ALL_URL = (
    "https://figshare.scilifelab.se/ndownloader/articles/28263911/versions/2"
)
DATASET_PAGE = (
    "https://figshare.scilifelab.se/articles/dataset/"
    "Single-cell_RNA_sequencing_data_on_primary_samples_from_"
    "Aberrant_expression_of_SLAMF6_constitutes_a_targetable_immune_escape_mechanism_in_acute_myeloid_leukemia/28263911"
)
LOCAL_H5AD_PATH = None  # e.g. r"data/annotated_AML_matrix.h5ad" to use a file you downloaded manually
AUTO_DOWNLOAD_DATASET = True

# Group configuration. Leave CELLTYPE_COL=None to use a best-effort common-column detector.
# After inspecting the printed metadata, specify the exact annotation column and exact label(s)
# if automatic matching does not select the populations you want.
CELLTYPE_COL = None
LEUKEMIA_LABELS = None  # e.g. ["AML blasts"] or ["Leukemic stem/progenitor"]
IMMUNE_LABELS = None    # e.g. ["CD8 T cells"] (recommended: one lineage at a time)
POOL_IMMUNE_LABELS = False  # False creates a separate network for each selected immune label
MAX_AUTO_IMMUNE_TYPES = 3  # limit automatic selection; raise this to examine more immune populations

# Compute settings: lower values are useful on laptops / limited RAM.
MAX_CELLS_PER_GROUP = 2500
MIN_CELLS_PER_GROUP = 100
N_HIGHLY_VARIABLE_TARGETS = 200
MAX_TFS = 100
TOP_K_REGULATORS_PER_TARGET = 5
N_TREES = 60
MAX_DEPTH = 12
MIN_SAMPLES_LEAF = 5

# Optional evaluation against a compatible gold-standard network CSV. Leave None to skip.
# Accepted edge column names include TF/target, regulator/target, or Gene1/Gene2.
GROUND_TRUTH_EDGE_PATH = None  # e.g. r"data/ground_truth_edges.csv"

print("Configuration ready.")

## 1. Download or locate the published AML dataset

The notebook first looks for a local `.h5ad` file. If none is present, it attempts to download the public Figshare article archive. The file is approximately 174 MB according to the repository page, so the download may take a few minutes. If automatic download is blocked by the repository, download the annotated `.h5ad` matrix from the dataset page and place it in `data/`, then rerun this section.

In [ ]:
def _download_figshare_archive(url: str, destination: Path) -> Path:
    """Download the public Figshare all-files archive with a helpful error on failure."""
    headers = {"User-Agent": "Mozilla/5.0 (compatible; GRN-analysis-notebook/1.0)"}
    with requests.get(url, stream=True, timeout=(30, 180), headers=headers, allow_redirects=True) as r:
        r.raise_for_status()
        content_type = r.headers.get("content-type", "").lower()
        first_chunk = None
        with open(destination, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                if first_chunk is None:
                    first_chunk = chunk[:16]
                f.write(chunk)
    if destination.stat().st_size < 1000:
        raise RuntimeError("The server returned a file that is too small to be the dataset archive.")
    return destination


def locate_h5ad() -> Path | None:
    if LOCAL_H5AD_PATH:
        p = Path(LOCAL_H5AD_PATH).expanduser()
        if p.exists() and p.suffix.lower() == ".h5ad":
            return p
        raise FileNotFoundError(f"LOCAL_H5AD_PATH does not exist or is not .h5ad: {p}")
    paths = sorted(DATA_DIR.rglob("*.h5ad"))
    return paths[0] if paths else None


def unpack_downloaded_archive(archive_path: Path, destination_dir: Path) -> None:
    """Unpack zip/tar archives safely into DATA_DIR; ignore non-archive responses."""
    try:
        if zipfile.is_zipfile(archive_path):
            with zipfile.ZipFile(archive_path) as zf:
                # Prevent path traversal from archive members.
                root = destination_dir.resolve()
                for member in zf.infolist():
                    target = (destination_dir / member.filename).resolve()
                    if root not in target.parents and target != root:
                        raise RuntimeError(f"Unsafe archive path rejected: {member.filename}")
                zf.extractall(destination_dir)
            return
        if tarfile.is_tarfile(archive_path):
            with tarfile.open(archive_path) as tf:
                root = destination_dir.resolve()
                for member in tf.getmembers():
                    target = (destination_dir / member.name).resolve()
                    if root not in target.parents and target != root:
                        raise RuntimeError(f"Unsafe archive path rejected: {member.name}")
                tf.extractall(destination_dir)
            return
        # If the server returned an HDF5 file rather than an archive, keep a .h5ad copy.
        with open(archive_path, "rb") as f:
            signature = f.read(8)
        if signature == b"\x89HDF\r\n\x1a\n":
            direct = destination_dir / "published_aml_matrix.h5ad"
            if archive_path.resolve() != direct.resolve():
                shutil.copy2(archive_path, direct)
            return
        raise RuntimeError("Downloaded response is neither a ZIP/TAR archive nor an HDF5 file.")
    except Exception:
        raise

h5ad_path = locate_h5ad()
if h5ad_path is None and AUTO_DOWNLOAD_DATASET:
    archive_path = DATA_DIR / "scilifelab_aml_dataset_download"
    try:
        print("Downloading the published AML dataset archive. This may take a few minutes...")
        _download_figshare_archive(FIGSHARE_DOWNLOAD_ALL_URL, archive_path)
        unpack_downloaded_archive(archive_path, DATA_DIR)
        h5ad_path = locate_h5ad()
    except Exception as exc:
        print(f"Automatic download did not succeed: {type(exc).__name__}: {exc}")
        print("Manual fallback: open the dataset page, download the annotated .h5ad matrix, and place it in the data/ folder:")
        print(DATASET_PAGE)

if h5ad_path is None:
    raise FileNotFoundError(
        "No .h5ad file was found. Download the annotated matrix from the Nature Cancer dataset page "
        f"and place it in '{DATA_DIR.resolve()}', or set LOCAL_H5AD_PATH to the file location.\n{DATASET_PAGE}"
    )

print(f"Using dataset file: {h5ad_path.resolve()}")

In [ ]:
# Load annotated single-cell matrix.
adata = ad.read_h5ad(h5ad_path)
adata.var_names_make_unique()
print(adata)
print(f"Cells: {adata.n_obs:,} | Genes/features: {adata.n_vars:,}")
print("Observation metadata columns:")
print(list(adata.obs.columns))

# Show likely categorical annotation fields and their common labels to guide group selection.
for col in adata.obs.columns:
    s = adata.obs[col]
    if (pd.api.types.is_categorical_dtype(s) or s.dtype == object or pd.api.types.is_string_dtype(s)):
        nunique = s.nunique(dropna=True)
        if 2 <= nunique <= 80:
            top = s.astype(str).value_counts().head(20)
            print(f"\n[{col}] unique={nunique}")
            display(top.rename("n_cells").to_frame())

## 2. Identify leukemic and immune cell populations

The code can identify common label patterns, but annotation names differ between studies. Inspect the metadata printed above. For a reproducible analysis, set `CELLTYPE_COL`, `LEUKEMIA_LABELS`, and `IMMUNE_LABELS` in the configuration cell to exact values from your data. Avoid pooling different immune lineages unless that is the explicit biological question.

In [ ]:
def choose_celltype_column(obs: pd.DataFrame) -> str | None:
    """Best-effort search for a cell annotation column; user override is preferred."""
    if CELLTYPE_COL is not None:
        if CELLTYPE_COL not in obs.columns:
            raise KeyError(f"CELLTYPE_COL={CELLTYPE_COL!r} not found. Available columns: {list(obs.columns)}")
        return CELLTYPE_COL
    ranked = []
    for col in obs.columns:
        name = str(col).lower().replace("-", "_").replace(" ", "_")
        nunique = obs[col].nunique(dropna=True)
        if not (2 <= nunique <= 150):
            continue
        if not (pd.api.types.is_categorical_dtype(obs[col]) or obs[col].dtype == object or pd.api.types.is_string_dtype(obs[col])):
            continue
        score = 0
        if "cell_type" in name or "celltype" in name: score += 10
        if "annotation" in name or "annot" in name: score += 5
        if "identity" in name or "label" in name: score += 3
        if "sample" in name or "patient" in name or "batch" in name: score -= 5
        if score > 0:
            ranked.append((score, col))
    ranked.sort(reverse=True)
    return ranked[0][1] if ranked else None

LEUKEMIA_PATTERN = re.compile(r"\b(aml|leuk|leukaem|leukem|blast|malignant|tumou?r|lsc)\b", re.I)
IMMUNE_PATTERN = re.compile(r"(\bt\s*cells?\b|\bcd[348] t\b|\bnk\b|\bb cells?\b|\bmonocytes?\b|\bmacrophages?\b|dendritic|neutrophil|lymphocyte|immune|plasma cell|granulocyte)", re.I)

celltype_col = choose_celltype_column(adata.obs)
if celltype_col is None:
    raise ValueError(
        "Could not automatically identify a cell annotation column. Review adata.obs.columns above, "
        "then set CELLTYPE_COL in the configuration cell and rerun this section."
    )
labels = adata.obs[celltype_col].astype(str)
all_labels = sorted(labels.dropna().unique().tolist())

if LEUKEMIA_LABELS is None:
    leukemia_labels = [x for x in all_labels if LEUKEMIA_PATTERN.search(x)]
else:
    leukemia_labels = list(LEUKEMIA_LABELS)
    missing = sorted(set(leukemia_labels) - set(all_labels))
    if missing:
        raise ValueError(f"These LEUKEMIA_LABELS were not found in {celltype_col}: {missing}")

if IMMUNE_LABELS is None:
    immune_candidates = [x for x in all_labels if IMMUNE_PATTERN.search(x) and x not in leukemia_labels]
    # Keep the largest annotated immune populations by cell count when automatic matching is used.
    immune_counts = labels.value_counts()
    immune_labels = sorted(immune_candidates, key=lambda x: int(immune_counts.get(x, 0)), reverse=True)[:MAX_AUTO_IMMUNE_TYPES]
else:
    immune_labels = list(IMMUNE_LABELS)
    missing = sorted(set(immune_labels) - set(all_labels))
    if missing:
        raise ValueError(f"These IMMUNE_LABELS were not found in {celltype_col}: {missing}")

print(f"Selected cell annotation column: {celltype_col}")
print("Detected/selected leukemic labels:", leukemia_labels)
print("Detected/selected immune labels:", immune_labels)
print("\nSelected label counts:")
selected_labels = list(dict.fromkeys(leukemia_labels + immune_labels))
display(labels[labels.isin(selected_labels)].value_counts().rename("n_cells").to_frame())

if not leukemia_labels:
    raise ValueError(
        "No leukemic labels were auto-detected. Set LEUKEMIA_LABELS to the exact annotation(s) printed above. "
        "Do not assume a cell is malignant based only on its lineage label."
    )
if not immune_labels:
    raise ValueError(
        "No immune labels were auto-detected. Set IMMUNE_LABELS to one or more exact immune-cell annotation values printed above."
    )

## 3. Prepare cells and expression features

A common gene/TF feature set is used across groups so that edge scores are more comparable. The default limits subsample each population for runtime; increase them for larger analyses. This notebook does not overwrite the source data.

In [ ]:
def to_log_normalized_matrix(adata_subset: ad.AnnData) -> sp.csr_matrix | np.ndarray:
    """Use counts layer if available; normalize and log-transform raw-like counts."""
    has_counts_layer = "counts" in adata_subset.layers
    X = adata_subset.layers["counts"] if has_counts_layer else adata_subset.X
    if sp.issparse(X):
        X = X.tocsr().astype(np.float32)
        data = X.data
        max_value = float(data.max()) if data.size else 0.0
        integer_like = (data.size > 0) and np.all(np.isclose(data[:min(data.size, 20000)] % 1, 0))
    else:
        X = np.asarray(X, dtype=np.float32)
        max_value = float(np.nanmax(X)) if X.size else 0.0
        flat = X.ravel()
        integer_like = (flat.size > 0) and np.all(np.isclose(flat[:min(flat.size, 20000)] % 1, 0))

    # If values appear to be raw counts, library-size normalize to 10,000 and apply log1p.
    # Otherwise presume the matrix is already normalized/log-transformed.
    needs_normalization = has_counts_layer or (integer_like and max_value > 0) or max_value > 30
    if not needs_normalization:
        return X

    if sp.issparse(X):
        totals = np.asarray(X.sum(axis=1)).ravel().astype(np.float64)
        factors = np.divide(1e4, totals, out=np.zeros_like(totals), where=totals > 0)
        Xn = sp.diags(factors.astype(np.float32)) @ X
        Xn = Xn.tocsr().astype(np.float32)
        Xn.data = np.log1p(Xn.data)
        return Xn
    totals = X.sum(axis=1, keepdims=True).astype(np.float64)
    factors = np.divide(1e4, totals, out=np.zeros_like(totals), where=totals > 0)
    return np.log1p(X * factors).astype(np.float32)

# Build group masks. AML/leukemic annotations are one combined group; immune annotations
# are separated by lineage unless POOL_IMMUNE_LABELS=True.
obs_labels = adata.obs[celltype_col].astype(str)
group_specs: dict[str, np.ndarray] = {}
leuk_mask = obs_labels.isin(leukemia_labels).to_numpy()
if leuk_mask.sum() >= MIN_CELLS_PER_GROUP:
    group_specs["AML_leukemic"] = leuk_mask
else:
    raise ValueError(f"Only {int(leuk_mask.sum())} cells selected for AML/leukemic; minimum is {MIN_CELLS_PER_GROUP}.")

if POOL_IMMUNE_LABELS:
    immune_mask = obs_labels.isin(immune_labels).to_numpy()
    if immune_mask.sum() >= MIN_CELLS_PER_GROUP:
        group_specs["Immune_pooled"] = immune_mask
    else:
        warnings.warn(f"Pooled immune group has only {int(immune_mask.sum())} cells; skipping.")
else:
    for label in immune_labels:
        mask = (obs_labels == label).to_numpy()
        safe_label = re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_")[:60] or "immune"
        if mask.sum() >= MIN_CELLS_PER_GROUP:
            group_specs[f"Immune_{safe_label}"] = mask
        else:
            warnings.warn(f"Skipping immune label {label!r}: only {int(mask.sum())} cells.")

if len(group_specs) < 2:
    raise ValueError("Need at least one AML/leukemic group and one immune group with sufficient cells.")

# Cap cell count per group before feature selection for tractable memory use.
selected_positions = {}
for group_name, mask in group_specs.items():
    idx = np.flatnonzero(mask)
    if len(idx) > MAX_CELLS_PER_GROUP:
        idx = rng.choice(idx, size=MAX_CELLS_PER_GROUP, replace=False)
    selected_positions[group_name] = np.sort(idx)
    print(f"{group_name}: {len(idx):,} cells")

all_idx = np.unique(np.concatenate(list(selected_positions.values())))
adata_selected = adata[all_idx].copy()
X_norm = to_log_normalized_matrix(adata_selected)
var_names = np.asarray(adata_selected.var_names.astype(str))

# Compute variances sparsely; no dense all-gene matrix is made here.
if sp.issparse(X_norm):
    means = np.asarray(X_norm.mean(axis=0)).ravel()
    means_sq = np.asarray(X_norm.power(2).mean(axis=0)).ravel()
else:
    means = X_norm.mean(axis=0)
    means_sq = np.square(X_norm).mean(axis=0)
variances = np.maximum(means_sq - means**2, 0)

print(f"Prepared pooled selected matrix: {X_norm.shape[0]:,} cells × {X_norm.shape[1]:,} genes")
print("Note: check the count/log-normalization assumption in to_log_normalized_matrix() if your input has unusual scaling.")

## 4. Build the candidate TF list and infer group-specific networks

The notebook attempts to download the human TF-symbol list distributed with pySCENIC. If internet access is unavailable, it uses a small, explicitly labelled fallback list for a demonstration run. For publication-grade analysis, use a current, comprehensive TF list and verify gene-symbol mapping (especially if your `.h5ad` uses Ensembl IDs).

In [ ]:
TF_LIST_URL = "https://raw.githubusercontent.com/aertslab/pySCENIC/master/resources/hs_hgnc_tfs.txt"
TF_LIST_FILE = DATA_DIR / "hs_hgnc_tfs.txt"
FALLBACK_TFS = set("""
AR AHR ARNT ATF1 ATF2 ATF3 ATF4 ATF5 ATF6 BATF BACH1 BACH2 BCL11A BCL6 BCLAF1 BHLHE40 BHLHE41 CEBPA CEBPB CEBPD CREB1 CREM E2F1 E2F2 E2F3 EBF1 EGR1 ELF1 ELK1 ETS1 ETS2 FOS FOSB FOXO1 FOXO3 FOXO4 GATA1 GATA2 GATA3 HIF1A IKZF1 IKZF2 IKZF3 IRF1 IRF2 IRF3 IRF4 IRF5 IRF7 IRF8 JUN JUNB JUND KLF2 KLF3 KLF4 KLF6 KLF9 LEF1 MAF MAFA MAFB MAX MEF2A MEF2C MITF MYB MYC MYCN NFE2 NFKB1 NFKB2 REL RELA RELB NR4A1 NR4A2 PAX5 PBX1 POU2F2 PRDM1 RUNX1 RUNX2 RUNX3 SPI1 SPIC STAT1 STAT2 STAT3 STAT4 STAT5A STAT5B STAT6 TAL1 TBX21 TCF3 TCF7 TFAP4 TP53 USF1 USF2 YY1 ZBTB7A ZEB1 ZEB2 ZNF143 SOX4 SOX11 SMAD3 SMAD4 NOTCH1 RBPJ HES1 HES4 ETV6 GFI1 GFI1B HOXA9 MEIS1 PBX3 WT1 CUX1 NFATC1 NFATC2 NFATC3 NFATC4 PRDM1 BCL11B FOXP1 FOXP3 BACH2 TOX TOX2 CIITA RUNX1T1
""".split())

def load_tf_symbols() -> set[str]:
    if not TF_LIST_FILE.exists():
        try:
            response = requests.get(TF_LIST_URL, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
            response.raise_for_status()
            TF_LIST_FILE.write_text(response.text, encoding="utf-8")
        except Exception as exc:
            warnings.warn(
                f"Could not retrieve the full pySCENIC TF list ({type(exc).__name__}: {exc}). "
                "Using a smaller fallback TF list. For final analyses, provide/download a comprehensive TF list."
            )
    if TF_LIST_FILE.exists():
        vals = [line.strip().upper() for line in TF_LIST_FILE.read_text(encoding="utf-8").splitlines()]
        vals = [v for v in vals if v and not v.startswith("#")]
        if vals:
            return set(vals)
    return FALLBACK_TFS

tf_symbols = load_tf_symbols()
upper_to_original = {}
for g in var_names:
    upper_to_original.setdefault(g.upper(), g)
expressed_tf_names = [upper_to_original[t] for t in tf_symbols if t in upper_to_original]

# Select shared variable targets and variable TFs from the pooled analysis cells.
variable_order = np.argsort(variances)[::-1]
variable_genes = [var_names[i] for i in variable_order if variances[i] > 1e-8]
tf_var_ordered = [g for g in variable_genes if g.upper() in tf_symbols]
selected_tfs = tf_var_ordered[:MAX_TFS]
selected_targets = []
for g in variable_genes:
    if g not in selected_targets:
        selected_targets.append(g)
    if len(selected_targets) >= N_HIGHLY_VARIABLE_TARGETS:
        break

if len(selected_tfs) < 10:
    warnings.warn(
        f"Only {len(selected_tfs)} candidate TFs matched gene names. Check if var_names are gene symbols rather than Ensembl IDs."
    )
if len(selected_targets) < 20:
    raise ValueError("Too few variable target genes were available after filtering.")

feature_genes = list(dict.fromkeys(selected_tfs + selected_targets))
feature_positions = {g: i for i, g in enumerate(var_names)}
feature_idx = np.array([feature_positions[g] for g in feature_genes], dtype=int)
X_features = X_norm[:, feature_idx]
if sp.issparse(X_features):
    X_features = X_features.toarray().astype(np.float32)
else:
    X_features = np.asarray(X_features, dtype=np.float32)

# Map global selected positions to rows in the pooled matrix.
row_lookup = {int(original_idx): local_idx for local_idx, original_idx in enumerate(all_idx)}
local_group_rows = {
    name: np.array([row_lookup[int(i)] for i in idx], dtype=int)
    for name, idx in selected_positions.items()
}
feature_to_col = {g: i for i, g in enumerate(feature_genes)}
regulator_genes = [g for g in selected_tfs if g in feature_to_col]
target_genes = [g for g in selected_targets if g in feature_to_col]

print(f"Candidate TFs: {len(regulator_genes)}")
print(f"Candidate target genes: {len(target_genes)}")
print("First 20 TFs:", regulator_genes[:20])
print("First 20 targets:", target_genes[:20])


def infer_extratrees_grn(X_group: np.ndarray, group_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return full TF-target importance scores and the top-K edge list for one group."""
    gene_col = {g: feature_to_col[g] for g in feature_genes}
    tf_cols = np.array([gene_col[g] for g in regulator_genes], dtype=int)
    X_tf_all = X_group[:, tf_cols]
    score_records = []
    top_records = []

    for target in tqdm(target_genes, desc=f"Inferring {group_name}", leave=False):
        y = X_group[:, gene_col[target]]
        if np.nanstd(y) < 1e-7:
            continue
        # Avoid trivial self-edges when a TF is also a target.
        allowed = np.array([tf != target for tf in regulator_genes], dtype=bool)
        if not allowed.any():
            continue
        model = ExtraTreesRegressor(
            n_estimators=N_TREES,
            max_depth=MAX_DEPTH,
            min_samples_leaf=MIN_SAMPLES_LEAF,
            max_features=0.8,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
        model.fit(X_tf_all[:, allowed], y)
        allowed_tfs = np.array(regulator_genes)[allowed]
        importances = model.feature_importances_
        for tf, score in zip(allowed_tfs, importances):
            score_records.append((tf, target, float(score), group_name))
        top_idx = np.argsort(importances)[::-1][:TOP_K_REGULATORS_PER_TARGET]
        for j in top_idx:
            top_records.append((allowed_tfs[j], target, float(importances[j]), group_name))

    score_df = pd.DataFrame(score_records, columns=["TF", "target", "importance", "group"])
    edge_df = pd.DataFrame(top_records, columns=["TF", "target", "importance", "group"])
    if not edge_df.empty:
        edge_df = edge_df.sort_values(["target", "importance"], ascending=[True, False]).reset_index(drop=True)
    return score_df, edge_df

all_scores = {}
all_edges = {}
for group_name, rows in local_group_rows.items():
    X_group = X_features[rows, :]
    score_df, edge_df = infer_extratrees_grn(X_group, group_name)
    all_scores[group_name] = score_df
    all_edges[group_name] = edge_df
    safe_group = re.sub(r"[^A-Za-z0-9_-]+", "_", group_name)
    edge_df.to_csv(OUTPUT_DIR / f"network_{safe_group}.csv", index=False)
    print(f"{group_name}: saved {len(edge_df):,} top-ranked edges")

print(f"\nOutputs written to: {OUTPUT_DIR.resolve()}")

## 5. Compare AML and immune networks

Importance scores are separately normalized by each fitted model. Differences are therefore useful for prioritization, but are not statistical significance tests. Treat `delta_importance` as a ranked hypothesis and follow up with biological validation.

In [ ]:
def make_differential_table(aml_scores: pd.DataFrame, immune_scores: pd.DataFrame,
                           aml_name: str, immune_name: str) -> pd.DataFrame:
    a = aml_scores[["TF", "target", "importance"]].rename(columns={"importance": "aml_importance"})
    b = immune_scores[["TF", "target", "importance"]].rename(columns={"importance": "immune_importance"})
    merged = a.merge(b, on=["TF", "target"], how="outer")
    merged[["aml_importance", "immune_importance"]] = merged[["aml_importance", "immune_importance"]].fillna(0.0)
    merged["delta_importance"] = merged["aml_importance"] - merged["immune_importance"]
    merged["abs_delta"] = merged["delta_importance"].abs()
    merged["higher_in"] = np.where(
        merged["delta_importance"] > 0, aml_name,
        np.where(merged["delta_importance"] < 0, immune_name, "equal")
    )
    return merged.sort_values("abs_delta", ascending=False).reset_index(drop=True)

aml_group_name = "AML_leukemic"
if aml_group_name not in all_scores:
    raise KeyError("AML/leukemic network was not inferred.")

comparison_tables = {}
for immune_group_name in [g for g in all_scores if g != aml_group_name]:
    diff = make_differential_table(
        all_scores[aml_group_name], all_scores[immune_group_name], aml_group_name, immune_group_name
    )
    comparison_tables[immune_group_name] = diff
    safe_immune = re.sub(r"[^A-Za-z0-9_-]+", "_", immune_group_name)
    diff.to_csv(OUTPUT_DIR / f"differential_AML_vs_{safe_immune}.csv", index=False)
    print(f"\nTop differential edges: AML vs {immune_group_name}")
    display(diff.head(15))

# Summarize top candidate regulators per group.
regulator_summary = []
for group_name, edge_df in all_edges.items():
    if edge_df.empty:
        continue
    temp = (edge_df.groupby("TF")
            .agg(n_top_edges=("target", "nunique"), mean_importance=("importance", "mean"),
                 total_importance=("importance", "sum"))
            .sort_values(["n_top_edges", "total_importance"], ascending=False)
            .head(25).reset_index())
    temp["group"] = group_name
    regulator_summary.append(temp)
if regulator_summary:
    top_regulators = pd.concat(regulator_summary, ignore_index=True)
    top_regulators.to_csv(OUTPUT_DIR / "top_regulators_by_group.csv", index=False)
    display(top_regulators.head(30))

## 6. Visualize top network edges

The plots display the strongest selected TF→target edges in each inferred network. To keep plots readable, only a small number of highest-importance edges are displayed.

In [ ]:
def plot_network(edge_df: pd.DataFrame, title: str, out_path: Path, n_edges: int = 45) -> None:
    if edge_df.empty:
        print(f"No edges to plot for {title}")
        return
    top = edge_df.nlargest(min(n_edges, len(edge_df)), "importance")
    G = nx.DiGraph()
    for row in top.itertuples(index=False):
        G.add_edge(row.TF, row.target, weight=float(row.importance))
    if G.number_of_nodes() > 75:
        # Keep only a smaller readable subset if the top edges still have many nodes.
        top = top.nlargest(min(30, len(top)), "importance")
        G = nx.DiGraph()
        for row in top.itertuples(index=False):
            G.add_edge(row.TF, row.target, weight=float(row.importance))
    pos = nx.spring_layout(G, seed=RANDOM_STATE, k=0.8)
    tf_nodes = [n for n in G.nodes if G.out_degree(n) > 0]
    target_only = [n for n in G.nodes if G.in_degree(n) > 0 and G.out_degree(n) == 0]
    weights = [1.0 + 5.0 * G[u][v]["weight"] / max((d["weight"] for _, _, d in G.edges(data=True)), default=1.0)
               for u, v in G.edges()]
    fig, ax = plt.subplots(figsize=(14, 10))
    nx.draw_networkx_nodes(G, pos, nodelist=tf_nodes, node_size=520, node_shape="o", alpha=0.9, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=target_only, node_size=380, node_shape="s", alpha=0.8, ax=ax)
    nx.draw_networkx_edges(G, pos, width=weights, arrows=True, arrowsize=12, alpha=0.45, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
    ax.set_title(title)
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

for group_name, edge_df in all_edges.items():
    safe_group = re.sub(r"[^A-Za-z0-9_-]+", "_", group_name)
    plot_network(edge_df, f"Inferred TF-target network: {group_name}", OUTPUT_DIR / f"network_{safe_group}.png")

## 7. Biological focus: candidate regulators linked to immune function / AML

This section does not force specific genes into the model. It extracts edges only when the regulator/target was measured and selected by the variable-gene/TF filters. The focus list is a transparent prioritization aid, not evidence that any listed edge is real.

In [ ]:
FOCUS_GENES = {
    "SLAMF6", "CIITA", "HLA-DRA", "HLA-DRB1", "HLA-DPA1", "HLA-DPB1", "B2M",
    "IRF1", "STAT1", "MYB", "CEBPA", "SPI1", "RUNX1", "GATA2", "NFKB1", "RELA",
    "CD3D", "CD3E", "CD8A", "CD4", "NKG7", "PRF1", "GZMB", "TOX", "TOX2", "PDCD1",
}

focus_rows = []
for group_name, edge_df in all_edges.items():
    if edge_df.empty:
        continue
    selected = edge_df[
        edge_df["TF"].astype(str).str.upper().isin(FOCUS_GENES)
        | edge_df["target"].astype(str).str.upper().isin(FOCUS_GENES)
    ].copy()
    if not selected.empty:
        selected["group"] = group_name
        focus_rows.append(selected)

if focus_rows:
    focus_edges = pd.concat(focus_rows, ignore_index=True)
    focus_edges.to_csv(OUTPUT_DIR / "focus_gene_edges.csv", index=False)
    display(focus_edges.sort_values("importance", ascending=False).head(50))
else:
    print("No focus-list gene appeared among the selected TFs/targets. This can happen because of expression filtering or gene-ID naming; inspect gene symbols and filters.")

## 8. Optional gold-standard evaluation

Set `GROUND_TRUTH_EDGE_PATH` in the configuration cell to a CSV containing directed TF→target edges. This can be a compatible reference network from a benchmark such as BEELINE or a curated network relevant to your selected gene set. The code reports average precision (AUPRC-style summary) and AUROC when both classes are available.

**Important:** a gold-standard network from a different cell type, organism, or gene-ID namespace is not a valid AML-specific truth set. Evaluation against the corresponding BEELINE synthetic/curated expression benchmark should be done using its matched expression matrix and its own reference network, rather than using AML cells with an unrelated ground truth.

In [ ]:
def load_ground_truth(path: str | Path) -> set[tuple[str, str]]:
    gt = pd.read_csv(path)
    lower = {c.lower(): c for c in gt.columns}
    tf_col = next((lower[k] for k in ["tf", "regulator", "gene1", "source"] if k in lower), None)
    target_col = next((lower[k] for k in ["target", "gene2", "target_gene", "destination"] if k in lower), None)
    if tf_col is None or target_col is None:
        raise ValueError(f"Could not identify TF/target columns in {list(gt.columns)}")
    return set(zip(gt[tf_col].astype(str).str.upper(), gt[target_col].astype(str).str.upper()))


def evaluate_scores(score_df: pd.DataFrame, ground_truth_edges: set[tuple[str, str]], label: str) -> dict:
    scores = score_df.copy()
    scores["TF_upper"] = scores["TF"].astype(str).str.upper()
    scores["target_upper"] = scores["target"].astype(str).str.upper()
    scores["label"] = [int((tf, tg) in ground_truth_edges) for tf, tg in zip(scores.TF_upper, scores.target_upper)]
    if scores["label"].nunique() < 2:
        return {"group": label, "n_candidate_edges": len(scores), "n_gold_edges_matched": int(scores.label.sum()),
                "average_precision": np.nan, "AUROC": np.nan,
                "note": "Only one class in the candidate pairs; metrics not defined."}
    return {"group": label, "n_candidate_edges": len(scores), "n_gold_edges_matched": int(scores.label.sum()),
            "average_precision": float(average_precision_score(scores.label, scores.importance)),
            "AUROC": float(roc_auc_score(scores.label, scores.importance)), "note": "Interpret only for a matching dataset/gene universe."}

if GROUND_TRUTH_EDGE_PATH:
    gt_path = Path(GROUND_TRUTH_EDGE_PATH)
    if not gt_path.exists():
        raise FileNotFoundError(f"GROUND_TRUTH_EDGE_PATH not found: {gt_path}")
    gt_edges = load_ground_truth(gt_path)
    eval_rows = [evaluate_scores(score_df, gt_edges, group_name) for group_name, score_df in all_scores.items()]
    evaluation = pd.DataFrame(eval_rows)
    evaluation.to_csv(OUTPUT_DIR / "ground_truth_evaluation.csv", index=False)
    display(evaluation)
else:
    print("Skipped: set GROUND_TRUTH_EDGE_PATH in the configuration cell to evaluate against a compatible reference network.")

## 9. Summary and reproducibility checklist

Before interpreting the inferred edges, record the following in your analysis notes or manuscript:

- Exact Nature dataset version, DOI, `.h5ad` file name, and licensing terms.
- `CELLTYPE_COL`, exact leukemic labels, exact immune labels, and whether immune lineages were pooled.
- Whether the count matrix was normalized/log-transformed by this notebook or was presumed preprocessed.
- Cell subsampling limits, TF list source, number of TFs/targets, model hyperparameters, and random seed.
- Whether ground-truth evaluation used a truly matched benchmark and the exact reference-network citation.
- Orthogonal validation of high-priority edges. TF feature importance can capture predictive association and indirect relationships, not necessarily physical TF binding.

### Output files

- `network_<group>.csv` — top-ranked TF→target candidate edges for each group.
- `differential_AML_vs_<immune>.csv` — edge-score contrasts between AML/leukemic cells and each immune population.
- `top_regulators_by_group.csv` — summary of TFs appearing most often among the selected edges.
- `network_<group>.png` — visual network summaries.
- `focus_gene_edges.csv` — subset of inferred edges touching the focus genes, when present.
- `ground_truth_evaluation.csv` — optional gold-standard metrics.

### References

- Sandén C. et al. *Nature Cancer* (2025). “Aberrant expression of SLAMF6 constitutes a targetable immune escape mechanism in acute myeloid leukemia.” [Dataset DOI](https://doi.org/10.17044/scilifelab.28263911) · [Article](https://www.nature.com/articles/s43018-025-01054-6)
- Pratapa A. et al. *Nature Methods* 17, 147–154 (2020). “Benchmarking algorithms for gene regulatory network inference from single-cell transcriptomic data.” [Article](https://doi.org/10.1038/s41592-019-0690-6) · [BEELINE benchmark data](https://doi.org/10.5281/zenodo.3378975)
- Duren Z. et al. *Nature Communications* 12, 4763 (2021). “Sc-compReg enables the comparison of gene regulatory networks between conditions using single-cell data.” [Article](https://doi.org/10.1038/s41467-021-25089-2)